In [ ]:
# Mock Data Generator (Smart Conveyor Monitoring System)
Simulates FB1 tag shape (Motor_Run, SystemReady, FaultActive, ItemCount,
TempAlarmHigh, TempAlarmLow) using a random-walk temperature model.
OPC-UA live feed is architecturally blocked (see project README) —
this generates realistic mock data so CSV logging / alarm detection /
dashboard can be built and tested now.

In [11]:
import random

def initial_state():
    """Starting values for one mock conveyor. Call once per machine."""
    return{
    "temp": 22.0,
    "item_count": 0,
    "fault_active": False,
    }

In [12]:
def next_reading(state, fault_chance=0.02, reset_fault=False):
    """
    Advance one mock conveyor by a single time step.
    Returns a NEW state dict — does not mutate the input.
    """
    fault_active = state["fault_active"]
    if reset_fault:
        fault_active = False

    step = random.uniform(-0.5, 0.5)
    new_temp = state["temp"] + step
    new_temp = max(0.0, min(95.0, new_temp)) # keep in physically plausible range

    temp_alarm_high = new_temp > 80.0
    temp_alarm_low = new_temp < 5.0

    if not fault_active and random.random() < fault_chance:
        fault_active = True

    motor_run = not fault_active
    item_count = state["item_count"]
    if motor_run:
        item_count += 1
        if item_count > 9999:
            item_count = 0


    system_ready = (not fault_active) and (not temp_alarm_high) and (not temp_alarm_low)

    return{
    "temp" : new_temp,
    "item_count" : item_count,
    "fault_active" : fault_active,
    "motor_run" : motor_run,
    "system_ready" : system_ready,
    "temp_alarm_high" : temp_alarm_high,
    "temp_alarm_low" : temp_alarm_low,
    }

In [16]:
state = initial_state()
for i in range(15):
    force_fault = 1.0 if i == 5 else 0.0
    reset = (i == 10)
    state = next_reading(state, fault_chance=force_fault, reset_fault=reset)
    print(f"{i:2d} temp={state['temp']:6.2f} count={state['item_count']:4d}"
            f"fault={state['fault_active']!s:5} ready={state['system_ready']!s:5}"
            f"hi={state['temp_alarm_high']!s:5} lo={state['temp_alarm_low']!s:5}")

 0 temp= 21.55 count=   1fault=False ready=True hi=False lo=False
 1 temp= 21.42 count=   2fault=False ready=True hi=False lo=False
 2 temp= 21.01 count=   3fault=False ready=True hi=False lo=False
 3 temp= 21.17 count=   4fault=False ready=True hi=False lo=False
 4 temp= 20.88 count=   5fault=False ready=True hi=False lo=False
 5 temp= 20.62 count=   5fault=True  ready=Falsehi=False lo=False
 6 temp= 20.70 count=   5fault=True  ready=Falsehi=False lo=False
 7 temp= 20.61 count=   5fault=True  ready=Falsehi=False lo=False
 8 temp= 20.51 count=   5fault=True  ready=Falsehi=False lo=False
 9 temp= 20.35 count=   5fault=True  ready=Falsehi=False lo=False
10 temp= 20.15 count=   6fault=False ready=True hi=False lo=False
11 temp= 20.27 count=   7fault=False ready=True hi=False lo=False
12 temp= 20.29 count=   8fault=False ready=True hi=False lo=False
13 temp= 20.39 count=   9fault=False ready=True hi=False lo=False
14 temp= 20.26 count=  10fault=False ready=True hi=False lo=False


In [ ]:
## Summary — Mock Data Generator
- Built `next_reading()`: random-walk temp (not fresh-random) so alarm
  detection can be tested against continuous, physically plausible data.
- State passed as dict in/out (not global) — enables multi-machine sim later.
- Fault latches until explicit `reset_fault=True` — matches FB1's
  reset-dominant E-Stop behavior.
- Temp clamped [0, 95] — prevents unbounded random-walk drift outside
  any real conveyor's operating range.
- Concept debt: `if __name__ == "__main__"` guard not yet explained in depth.
- Next: wire next_reading() into a loop + CSV writer (Day 5).